# DIOR Faster R-CNN Training

Train one epoch at a time and download the checkpoint and log after each epoch.


In [ ]:
# Load the project source into the Colab runtime.
!test -d /content/ecs174-dior/.git || git clone https://github.com/viveknarayana/ecs174-dior.git /content/ecs174-dior
%cd /content/ecs174-dior
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Pull the latest project updates before using the cloned source.
!git -C /content/ecs174-dior pull origin main


In [ ]:
# Download the DIOR archives and extract them into the directory structure expected by DIORDataset.
!pip -q install gdown matplotlib
!mkdir -p /content/dior_archives /content/DIOR
!gdown --folder 'https://drive.google.com/drive/folders/1UdlgHk49iu6WpcJ5467iT-UqNPpx__CC' -O /content/dior_archives

from pathlib import Path
from zipfile import ZipFile
archive_dir = Path('/content/dior_archives')
data_root = Path('/content/DIOR')
for archive in sorted(archive_dir.glob('*.zip')):
    print('Extracting', archive.name)
    with ZipFile(archive) as zf:
        zf.extractall(data_root)

assert (data_root / 'JPEGImages-trainval').is_dir(), 'JPEGImages-trainval folder not found after extraction'
assert (data_root / 'JPEGImages-test').is_dir(), 'JPEGImages-test folder not found after extraction'
assert (data_root / 'Annotations' / 'Horizontal Bounding Boxes').is_dir(), 'Annotations folder not found after extraction'
assert (data_root / 'ImageSets' / 'Main' / 'train.txt').is_file() or (data_root / 'Main' / 'train.txt').is_file(), 'train.txt split file not found after extraction'
print('DIOR folder layout looks ready:', data_root)

In [ ]:
# Train one epoch per execution. Download files in the separate cell below.
import importlib
import csv
import subprocess
import sys
from pathlib import Path
import torch
import config
from google.colab import files
CONFIG = importlib.reload(config).CONFIG
OUTPUT_ROOT = '/content/dior_outputs'

def run_one_epoch(backbone='resnet50', resume=None, tiny=False):
    run_dir = Path(OUTPUT_ROOT) / backbone
    checkpoint = Path(resume) if resume else None
    if checkpoint is None and list(run_dir.glob('checkpoint_epoch_*.pth')):
        raise ValueError('Existing checkpoints found. Pass a checkpoint to resume or choose a fresh output folder.')
    previous_epoch = 0
    if checkpoint:
        saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
        previous_epoch = int(saved['epoch'])
        if not (run_dir / 'training_log.csv').is_file():
            raise ValueError('Restore the matching training log before resuming.')
        with (run_dir / 'training_log.csv').open(newline='') as f:
            rows = list(csv.DictReader(f))
        if not rows or int(rows[-1]['epoch']) != previous_epoch:
            raise ValueError('The CSV log must end at the checkpoint epoch.')
        print(f'Resuming from epoch {previous_epoch}.', flush=True)
        del saved
    next_epoch = previous_epoch + 1
    if next_epoch > 10:
        raise ValueError('The 10-epoch target is already complete.')
    command = [sys.executable, '-u', 'train.py', '--data-root', '/content/DIOR',
               '--backbone', backbone, '--output-root', OUTPUT_ROOT,
               '--epochs', str(next_epoch), '--batch-size', '2']
    if checkpoint:
        command += ['--resume', str(checkpoint)]
    if tiny:
        command += ['--tiny-test', '--tiny-train-images', '2', '--tiny-validation-images', '1']
    subprocess.run(command, check=True)
    print(f'Epoch {next_epoch} complete. Run the download cell next.', flush=True)

def download_epoch(backbone='resnet50', epoch=1):
    run_dir = Path(OUTPUT_ROOT) / backbone
    for name in (f'checkpoint_epoch_{epoch:02d}.pth', 'training_log.csv', 'loss_curves.png'):
        path = run_dir / name
        if not path.is_file():
            raise FileNotFoundError(path)
        files.download(str(path))


## Smoke test

The smoke test runs separately from the full experiment.


In [ ]:
# Run a tiny smoke test from the command line with a separate output folder.
!python train.py --data-root /content/DIOR --backbone resnet50 --output-root /content/smoke_test_outputs --tiny-test --tiny-train-images 2 --epochs 1 --batch-size 2


In [ ]:
# Inspect predictions from the smoke-test checkpoint on one training image.
import sys
sys.path.insert(0, '/content/ecs174-dior/DIOR_Data')
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from dataset import DIORDataset
from train import build_model
from classes import CLASS_NAMES

checkpoint_path = '/content/smoke_test_outputs/resnet50/checkpoint_epoch_01.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model = build_model(checkpoint['backbone'], checkpoint['config']['image_size'])
model.load_state_dict(checkpoint['model_state_dict'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()
dataset = DIORDataset('/content/DIOR', split='train')
image, target = dataset[0]
with torch.inference_mode():
    prediction = model([image.to(device)])[0]
print('Checkpoint epoch:', checkpoint['epoch'], 'Image ID:', dataset.ids[0])
print('Top five predicted classes and confidence scores:')
for score, label in zip(prediction['scores'][:5].cpu(), prediction['labels'][:5].cpu()):
    print(f'{CLASS_NAMES[label.item()-1]}: {score.item():.3f}')
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image.permute(1, 2, 0))
for box, score, label in zip(prediction['boxes'].cpu(), prediction['scores'].cpu(), prediction['labels'].cpu()):
    if score < 0.25:
        continue
    x1, y1, x2, y2 = box.tolist()
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='red', linewidth=2))
    ax.text(x1, y1, f'{CLASS_NAMES[label.item()-1]} {score:.2f}', color='white', backgroundcolor='red')
ax.set_title('Smoke-test predictions, confidence ≥ 0.25')
ax.axis('off')
plt.show()


## Full experiments

Run one training epoch, then run the separate download cell. Confirm the files arrived before continuing. Both backbones use the combined train and validation splits.


In [ ]:
# Start ResNet-50 epoch 1 in a fresh output folder.
run_one_epoch('resnet50')


In [ ]:
# Download the checkpoint, log, and plot after the training cell finishes.
download_epoch('resnet50', epoch=1)


## Resume from a downloaded checkpoint

Upload exactly one checkpoint and its matching CSV log. Then run the next-epoch cell.


In [ ]:
# Upload a checkpoint and matching CSV log from your computer.
from google.colab import files
import shutil
uploaded = files.upload()
checkpoints = [name for name in uploaded if name.endswith('.pth')]
logs = [name for name in uploaded if name.endswith('.csv')]
if len(checkpoints) != 1 or len(logs) != 1:
    raise ValueError('Upload exactly one checkpoint and one matching CSV log.')
run_dir = Path(OUTPUT_ROOT) / 'resnet50'
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(logs[0], run_dir / 'training_log.csv')
resume_checkpoint = str(Path(checkpoints[0]).resolve())
print('Checkpoint ready:', resume_checkpoint)


In [ ]:
# Continue for one epoch after uploading the checkpoint and CSV.
run_one_epoch('resnet50', resume=resume_checkpoint)
